# Feedforward Networks — MLP and Autoencoders in PyTorch

**Author:** Shivani Bokka  
**Datasets:** Adult Income (MLP tabular classification), MNIST digits (Autoencoder)  
**Goal:** Build proper PyTorch MLP and Autoencoder — understand every training component

---

## What Is This Notebook About?

This notebook is a **complete, from-scratch walkthrough of feedforward neural networks** built in PyTorch. We start with the fundamentals — activation functions, gradient problems — and work our way up to a full training pipeline, diagnostic tools, and finally an autoencoder for unsupervised learning.

Every section explains the **core idea in plain language** before diving into code. Every chart has a dedicated explanation so you know exactly what you are looking at.

---

## What Is a Feedforward Network?

A **feedforward neural network** (also called a **Multilayer Perceptron**, or MLP) is the simplest kind of neural network. Information flows in one direction — from input to output — through a series of layers. Each layer transforms its input and passes the result to the next layer.

Think of it like an assembly line:  
> Raw input goes in one end. Each station (layer) does something to it. A final prediction comes out the other end.

---

## Topics Covered

| # | Section | Key Idea |
|---|---------|----------|
| 1 | Imports and Setup | Libraries, seeds, device check |
| 2 | Activation Functions | ReLU, Sigmoid, Tanh, Leaky ReLU and their derivatives |
| 3 | Vanishing Gradient Problem | Why sigmoid kills deep networks, how ReLU fixes it |
| 4 | Adult Income Dataset | Load, preprocess, explore tabular data |
| 5 | Build MLP in PyTorch | nn.Module, forward pass, BatchNorm, Dropout |
| 6 | Weight Initialization | Xavier vs He — why it matters |
| 7 | Training Loop | Adam, CrossEntropyLoss, early stopping, learning curves |
| 8 | Diagnosing Fit | Underfitting vs overfitting vs good fit |
| 9 | BatchNorm and Dropout Effects | Regularization comparison table |
| 10 | MLP vs XGBoost | Tabular data benchmark |
| 11 | Autoencoder on MNIST | Compression, reconstruction, latent space |
| 12 | Anomaly Detection Preview | One-class training, reconstruction error |
| 13 | Summary and Key Takeaways | When to use what, common mistakes |

---

---

## Section 1 — Imports and Setup

We import everything we will need throughout this notebook in one place.

- **torch** — the core PyTorch library
- **torch.nn** — neural network layers and loss functions
- **torch.optim** — optimizers (Adam, SGD, etc.)
- **torch.utils.data** — DataLoader and TensorDataset for batching
- **torchvision** — standard vision datasets including MNIST
- **sklearn** — preprocessing, datasets, metrics
- **numpy / pandas** — data manipulation
- **matplotlib / seaborn** — visualization

We also set **random seeds** so our results are reproducible. Without fixed seeds, neural network training has randomness from weight initialization and data shuffling — the same code would give slightly different results every run.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import time
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Torchvision for MNIST
import torchvision
import torchvision.transforms as transforms

# Sklearn
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.decomposition import PCA

# XGBoost
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not installed — Section 10 will use a sklearn GradientBoostingClassifier instead.")
    from sklearn.ensemble import GradientBoostingClassifier

# Display settings
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")
print("All imports successful!")

---

## Section 2 — Activation Functions

### Why Do We Need Activation Functions?

**Activation functions introduce non-linearity — without them, stacking linear layers is just one linear layer.**

Here is why that matters: a linear function applied to a linear function is still a linear function. No matter how many layers you stack, if every layer does `output = weights * input + bias`, the whole network can only learn straight-line relationships — no curves, no complex patterns.

An activation function is applied after each linear layer. It squashes, clips, or distorts the output in a non-linear way, which allows the network to learn highly complex patterns.

### The Four Main Activation Functions

| Function | Formula | Output Range | Best Used For |
|----------|---------|-------------|---------------|
| **ReLU** | max(0, x) | [0, ∞) | Default choice for hidden layers |
| **Sigmoid** | 1 / (1 + e^-x) | (0, 1) | Binary output layer |
| **Tanh** | (e^x - e^-x) / (e^x + e^-x) | (-1, 1) | Hidden layers (historically) |
| **Leaky ReLU** | max(0.01x, x) | (-∞, ∞) | Fixes the dying ReLU problem |

**Plain English intuition:**
- **ReLU** (Rectified Linear Unit): "If the number is positive, keep it. If negative, make it zero." Simple, fast, effective.
- **Sigmoid**: "Squeeze any number into the range (0, 1)." Looks like an S-curve. Originally popular for hidden layers; now mainly used for binary outputs.
- **Tanh**: "Squeeze any number into the range (-1, 1)." A scaled version of sigmoid, centered at zero. Better than sigmoid for hidden layers but still has gradient problems.
- **Leaky ReLU**: "If positive, keep it. If negative, keep a tiny fraction of it (e.g., 1%)." Fixes ReLU's biggest problem.

In [ ]:
# Plot activation functions and their derivatives
x = np.linspace(-4, 4, 400)

# Activation functions
relu       = np.maximum(0, x)
sigmoid    = 1 / (1 + np.exp(-x))
tanh       = np.tanh(x)
leaky_relu = np.where(x >= 0, x, 0.01 * x)

# Derivatives
d_relu       = np.where(x >= 0, 1.0, 0.0)
d_sigmoid    = sigmoid * (1 - sigmoid)
d_tanh       = 1 - tanh ** 2
d_leaky_relu = np.where(x >= 0, 1.0, 0.01)

names   = ['ReLU', 'Sigmoid', 'Tanh', 'Leaky ReLU']
funcs   = [relu, sigmoid, tanh, leaky_relu]
derivs  = [d_relu, d_sigmoid, d_tanh, d_leaky_relu]
colors  = ['steelblue', 'tomato', 'seagreen', 'darkorchid']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Activation Functions (top) and Their Derivatives (bottom)', fontsize=14, fontweight='bold')

for i, (name, fn, dfn, color) in enumerate(zip(names, funcs, derivs, colors)):
    # Activation
    axes[0, i].plot(x, fn, color=color, linewidth=2.5)
    axes[0, i].axhline(0, color='gray', linewidth=0.8, linestyle='--')
    axes[0, i].axvline(0, color='gray', linewidth=0.8, linestyle='--')
    axes[0, i].set_title(name, fontweight='bold')
    axes[0, i].set_ylim(-1.5, 3)
    axes[0, i].set_xlabel('x')
    axes[0, i].set_ylabel('f(x)')
    axes[0, i].grid(True, alpha=0.4)

    # Derivative
    axes[1, i].plot(x, dfn, color=color, linewidth=2.5, linestyle='-.')
    axes[1, i].axhline(0, color='gray', linewidth=0.8, linestyle='--')
    axes[1, i].axvline(0, color='gray', linewidth=0.8, linestyle='--')
    axes[1, i].set_title(f"d/dx {name}", fontweight='bold')
    axes[1, i].set_ylim(-0.3, 1.3)
    axes[1, i].set_xlabel('x')
    axes[1, i].set_ylabel("f'(x)")
    axes[1, i].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

### How to Read This Chart: Activation Functions and Derivatives

This is a 2-row, 4-column grid of plots. The **top row** shows each activation function, and the **bottom row** shows its derivative (the gradient).

- **The x-axis** represents the input to the activation function — this is the raw value coming out of a linear layer.
- **The top row y-axis** is the output value after applying the activation.
- **The bottom row y-axis** is the gradient — how steeply the output changes as the input changes. This gradient is what flows backward during training.

**What to look for in the derivatives:**
- **ReLU's derivative** is either 0 (when x < 0) or 1 (when x > 0). This is the key to why ReLU works well — the gradient is always either zero or a constant, never shrinking exponentially.
- **Sigmoid's derivative** is always very small (maximum is 0.25 at x = 0). This means gradients get multiplied by at most 0.25 per layer — causing the **vanishing gradient problem**.
- **Tanh's derivative** is slightly better than sigmoid (maximum is 1.0 at x = 0), but still becomes very small away from zero.
- **Leaky ReLU's derivative** is either 0.01 (when x < 0) or 1 (when x > 0). Even for negative inputs, some small gradient flows through.

---

### The Dying ReLU Problem

ReLU has one significant weakness: the **dying neuron problem**.

Here is what happens:
1. A neuron receives negative input from every training example.
2. ReLU outputs 0 for all of them.
3. The **gradient is also 0** — the derivative of ReLU for negative inputs is 0.
4. Because the gradient is 0, the weights connecting to this neuron **never get updated during backpropagation**.
5. The neuron is permanently "dead" — it outputs 0 forever and contributes nothing to learning.

This can happen when:
- Learning rate is too large and a big weight update pushes inputs permanently negative.
- Weight initialization is poorly chosen.

**Leaky ReLU fixes this** by allowing a small gradient (0.01) for negative inputs. Even if a neuron gets stuck with negative inputs, it can still receive gradient signal and potentially recover.

In practice:
- Start with **ReLU** — it works well for most networks.
- Switch to **Leaky ReLU** if you observe dying neurons (neurons that never activate on any training example).
- Avoid **Sigmoid and Tanh** for hidden layers in deep networks due to vanishing gradients.

---

## Section 3 — The Vanishing Gradient Problem

**In deep networks with sigmoid activations, gradients shrink exponentially as they travel backward through layers. By layer 1, the gradient is near zero — the layer barely learns.**

### How Backpropagation Works (Plain English)

After a forward pass (input → output → loss), PyTorch calculates the **gradient of the loss with respect to every weight** using the chain rule. This travels backward through the network — from output layer to input layer.

At each layer, the gradient gets **multiplied** by:
1. The derivative of the activation function.
2. The current weights.

**The problem with sigmoid:** The maximum value of the sigmoid derivative is 0.25 (at x = 0). So at each layer, the gradient is multiplied by something ≤ 0.25. In a 10-layer network:

> 0.25 × 0.25 × 0.25 × ... (10 times) = 0.25^10 ≈ 0.000001

The gradient has shrunk by a factor of a million. The first few layers receive near-zero gradient and **barely update their weights**. They essentially don't learn.

**The fix:** Use ReLU, whose derivative is either 0 or 1 — gradients don't shrink as they travel back through ReLU layers. Add BatchNorm to further stabilize gradient flow.

In [ ]:
# ── Experiment 1: 10-layer MLP with Sigmoid activations ──
torch.manual_seed(42)

class DeepSigmoidNet(nn.Module):
    def __init__(self, depth=10, width=64):
        super().__init__()
        layers = [nn.Linear(width, width), nn.Sigmoid()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Sigmoid()]
        layers += [nn.Linear(width, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def get_gradient_norms(model, x, y):
    """Run one forward + backward pass and return gradient norms per linear layer."""
    model.zero_grad()
    output = model(x)
    loss = nn.MSELoss()(output, y)
    loss.backward()
    norms = []
    for module in model.modules():
        if isinstance(module, nn.Linear):
            if module.weight.grad is not None:
                norms.append(module.weight.grad.norm().item())
    return norms

# Dummy input
x_dummy = torch.randn(128, 64)
y_dummy = torch.randn(128, 1)

sigmoid_net = DeepSigmoidNet(depth=10, width=64)
sigmoid_norms = get_gradient_norms(sigmoid_net, x_dummy, y_dummy)

print("Sigmoid network — gradient norms per layer (layer 0 = last, layer 10 = first):")
for i, n in enumerate(sigmoid_norms):
    label = 'output' if i == 0 else f'layer {len(sigmoid_norms) - 1 - i}'
    print(f"  {label:>10}: {n:.8f}")

In [ ]:
# ── Experiment 2: 10-layer MLP with ReLU activations ──
torch.manual_seed(42)

class DeepReLUNet(nn.Module):
    def __init__(self, depth=10, width=64):
        super().__init__()
        layers = [nn.Linear(width, width), nn.ReLU()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.ReLU()]
        layers += [nn.Linear(width, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

relu_net = DeepReLUNet(depth=10, width=64)
relu_norms = get_gradient_norms(relu_net, x_dummy, y_dummy)

print("ReLU network — gradient norms per layer:")
for i, n in enumerate(relu_norms):
    label = 'output' if i == 0 else f'layer {len(relu_norms) - 1 - i}'
    print(f"  {label:>10}: {n:.8f}")

In [ ]:
# ── Experiment 3: ReLU + BatchNorm ──
torch.manual_seed(42)

class DeepReLUBNNet(nn.Module):
    def __init__(self, depth=10, width=64):
        super().__init__()
        layers = [nn.Linear(width, width), nn.BatchNorm1d(width), nn.ReLU()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.BatchNorm1d(width), nn.ReLU()]
        layers += [nn.Linear(width, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

relu_bn_net = DeepReLUBNNet(depth=10, width=64)
relu_bn_norms = get_gradient_norms(relu_bn_net, x_dummy, y_dummy)

print("ReLU + BatchNorm network — gradient norms per layer:")
for i, n in enumerate(relu_bn_norms):
    label = 'output' if i == 0 else f'layer {len(relu_bn_norms) - 1 - i}'
    print(f"  {label:>10}: {n:.8f}")

In [ ]:
# ── Plot all three side by side ──
n_layers = len(sigmoid_norms)
layer_labels = [f'L{i}' for i in range(n_layers - 1, -1, -1)]
layer_labels[-1] = 'Out'

x_pos = np.arange(n_layers)
width = 0.28

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x_pos - width,     sigmoid_norms,  width=width, label='Sigmoid',        color='tomato',     alpha=0.85)
ax.bar(x_pos,             relu_norms,     width=width, label='ReLU',           color='steelblue',  alpha=0.85)
ax.bar(x_pos + width,     relu_bn_norms,  width=width, label='ReLU + BatchNorm',color='seagreen',  alpha=0.85)

ax.set_xticks(x_pos)
ax.set_xticklabels(layer_labels)
ax.set_xlabel('Layer (L0 = closest to input, Out = output layer)')
ax.set_ylabel('Gradient L2 Norm (log scale)')
ax.set_yscale('log')
ax.set_title('Gradient Norms by Layer — Sigmoid vs ReLU vs ReLU + BatchNorm', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

### How to Read This Chart: Gradient Norms by Layer

This is a **grouped bar chart** where each group represents one layer of a 10-layer network. The x-axis runs from the input-side layers (L0) to the output layer (Out).

- **The y-axis** is the **L2 norm of the gradient** in that layer, shown on a **log scale**. The L2 norm is a single number summarizing how large the gradient vector is. A larger norm means the layer is receiving a stronger learning signal.
- **Red bars (Sigmoid):** The gradient norms collapse toward the input layers. By L0, the norm is nearly zero — those layers are not learning.
- **Blue bars (ReLU):** Gradient norms are much more consistent across all layers. The signal travels all the way back to the input layers.
- **Green bars (ReLU + BatchNorm):** Even more uniform. BatchNorm normalizes the activations at each layer, which keeps gradient flow stable throughout training.

**What near-zero in early layers means:**
A near-zero gradient norm in layer L0 means the weight update for that layer during this step is almost zero. If this happens every step, that layer never learns. It's as if those layers don't exist.

**The fix:**
- Use **ReLU** instead of Sigmoid for hidden layers.
- Add **BatchNorm** after each linear layer for additional stability.
- With these two changes, gradients flow cleanly through even very deep networks.

---

## Section 4 — Load Adult Income Dataset

### What Is This Dataset?

The **Adult Income dataset** (also called the Census Income dataset) comes from the 1994 UCI Machine Learning Repository. Each row represents a person. The goal is to predict whether that person earns more than $50,000 per year.

This is a classic **binary classification** problem on **tabular data** — the kind of problem where MLPs are commonly benchmarked against tree-based methods.

### Features:

| Feature | Type | Description |
|---------|------|-------------|
| age | Numeric | Age of the person |
| workclass | Categorical | Type of employer |
| fnlwgt | Numeric | Census sampling weight |
| education | Categorical | Highest education level |
| education-num | Numeric | Education years (numeric version) |
| marital-status | Categorical | Marital status |
| occupation | Categorical | Type of occupation |
| relationship | Categorical | Family relationship |
| race | Categorical | Race |
| sex | Categorical | Sex |
| capital-gain | Numeric | Investment gains |
| capital-loss | Numeric | Investment losses |
| hours-per-week | Numeric | Hours worked per week |
| native-country | Categorical | Country of origin |

### Target:
- **income** — `<=50K` (0) or `>50K` (1)

In [ ]:
# Load the Adult dataset from OpenML
print("Loading Adult Income dataset from OpenML (this may take a moment)...")
adult = fetch_openml('adult', version=2, as_frame=True, parser='auto')
df = adult.frame.copy()
print(f"Shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['class'].value_counts())
df.head()

In [ ]:
# ── Preprocessing ──
# Drop rows with missing values
df = df.dropna()

# Separate features and target
y_raw = df['class'].copy()
X_raw = df.drop('class', axis=1).copy()

# Encode target: <=50K → 0, >50K → 1
y_enc = (y_raw.astype(str).str.strip() == '>50K').astype(int).values

# Encode categorical features with LabelEncoder
cat_cols = X_raw.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()

X_encoded = X_raw.copy()
le = LabelEncoder()
for col in cat_cols:
    X_encoded[col] = le.fit_transform(X_raw[col].astype(str))

X_encoded = X_encoded.astype(float)

# Train/test split
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_encoded.values, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# Scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_np)
X_test_sc  = scaler.transform(X_test_np)

INPUT_DIM = X_train_sc.shape[1]

print(f"Training set  : {X_train_sc.shape}")
print(f"Test set      : {X_test_sc.shape}")
print(f"Input features: {INPUT_DIM}")
print(f"Class balance (train) — 0: {(y_train_np==0).sum()}, 1: {(y_train_np==1).sum()}")

In [ ]:
# Class balance bar chart
labels = ['<=50K (0)', '>50K (1)']
counts = [(y_enc == 0).sum(), (y_enc == 1).sum()]

plt.figure(figsize=(6, 4))
sns.barplot(x=labels, y=counts, palette=['steelblue', 'tomato'])
plt.title('Class Balance — Adult Income Dataset', fontweight='bold')
plt.ylabel('Count')
plt.xlabel('Income Class')
for i, v in enumerate(counts):
    plt.text(i, v + 100, f'{v:,}  ({v/len(y_enc)*100:.1f}%)', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

### How to Read This Chart: Class Balance

This is a simple **bar chart** showing how many examples belong to each class.

- **Blue bar** = people earning ≤ $50K per year (majority class)
- **Red bar** = people earning > $50K per year (minority class)

**Why this matters:**

This is an **imbalanced dataset** — there are roughly 3× more low-income examples than high-income examples. A model that always predicts "≤50K" would get ~75% accuracy without learning anything. This is called the **accuracy paradox**.

To properly evaluate a model on imbalanced data:
- Look at **F1 score** (harmonic mean of precision and recall) in addition to accuracy.
- Be skeptical of high accuracy — check whether the model is actually learning the minority class.

---

## Section 5 — Build the MLP in PyTorch (nn.Module)

### How PyTorch Models Work

Every PyTorch model is a Python class that inherits from `nn.Module`. You must define two methods:

**`__init__`** — called once when you create the model. This is where you define all the layers (Linear, BatchNorm, Dropout, etc.). Think of it as building the pieces of the assembly line before turning it on.

**`forward`** — called every time you pass data through the model. This defines the actual computation: what order do the layers run in, what goes into each layer, what comes out? PyTorch uses this to build the **computation graph** that backpropagation will trace.

### Architecture

Our MLP architecture is:

```
Input (14 features)
    → Linear(14 → 256) → BatchNorm1d(256) → ReLU → Dropout(0.3)
    → Linear(256 → 128) → BatchNorm1d(128) → ReLU → Dropout(0.3)
    → Linear(128 → 64)  → BatchNorm1d(64)  → ReLU → Dropout(0.3)
    → Linear(64 → 2)
    → Output (2 classes: 0 or 1)
```

**Why these design choices?**
- **Wider early layers, narrower later layers** — a common "funnel" pattern that compresses features hierarchically.
- **BatchNorm before activation** — normalizes the layer's output to have mean ≈ 0, std ≈ 1 before passing through ReLU.
- **Dropout** — randomly zeros out neurons during training to prevent co-adaptation.
- **Output size = 2** — because CrossEntropyLoss expects raw scores (logits) for each class. The model does not apply softmax — CrossEntropyLoss handles that internally.

In [ ]:
class MLP(nn.Module):
    """
    Multilayer Perceptron for binary classification.
    Architecture: input → 256 → 128 → 64 → 2
    Each hidden layer: Linear → BatchNorm1d → ReLU → Dropout
    """
    def __init__(self, input_dim, dropout_rate=0.3):
        super(MLP, self).__init__()

        self.layer1 = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.layer2 = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.layer3 = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.output_layer = nn.Linear(64, 2)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.output_layer(x)
        return x


# Instantiate and inspect
model = MLP(input_dim=INPUT_DIM).to(device)
print(model)

In [ ]:
# Manual parameter count
print("Model parameter summary:\n")
print(f"{'Layer':<35} {'Shape':<25} {'Parameters':>12}")
print("-" * 75)
total_params = 0
trainable_params = 0
for name, param in model.named_parameters():
    n = param.numel()
    total_params += n
    if param.requires_grad:
        trainable_params += n
    print(f"{name:<35} {str(list(param.shape)):<25} {n:>12,}")
print("-" * 75)
print(f"{'Total parameters':<35} {'':25} {total_params:>12,}")
print(f"{'Trainable parameters':<35} {'':25} {trainable_params:>12,}")

### How to Read This Output: Model Summary

The parameter summary table shows every set of trainable weights in the model.

- **Layer column** — the name of the parameter in PyTorch's naming convention. For example, `layer1.0.weight` means: `layer1` block → first submodule (index 0, which is the `nn.Linear`) → its weight matrix.
- **Shape column** — the dimensions of the parameter tensor. A `[256, 14]` weight matrix connects 14 input features to 256 neurons.
- **Parameters column** — total number of individual numbers in that tensor. For `[256, 14]`, that's 256 × 14 = 3,584 numbers.

**How to compute parameters manually:**
- A `Linear(in, out)` layer has `in × out` weight parameters plus `out` bias parameters.
- A `BatchNorm1d(n)` layer has `n` gamma (scale) and `n` beta (shift) parameters — 2n total.

**Total trainable parameters** = all the numbers the optimizer will adjust during training. More parameters = more expressive model, but also higher risk of overfitting and slower training.

---

## Section 6 — Weight Initialization

### Why Weight Initialization Matters

**Bad weight initialization can cause vanishing or exploding gradients from the very first forward pass** — before any training has even started.

If all weights start at zero: every neuron in a layer computes the same output → every gradient is the same → symmetry is never broken → the network can never learn different features in different neurons.

If weights start too large: activations grow exponentially through layers → numbers overflow → NaN losses.

If weights start too small: activations shrink exponentially through layers → vanishing gradients from step one.

### Xavier (Glorot) Initialization — for Sigmoid and Tanh

**Formula:** `Var(W) = 2 / (fan_in + fan_out)`

Where `fan_in` = number of inputs to a layer, `fan_out` = number of outputs.

**Plain English:** Xavier sets the weight variance so that the variance of activations stays roughly constant as data flows forward, and the variance of gradients stays roughly constant as they flow backward. It was derived assuming linear activations (or Tanh, which is approximately linear near zero).

### He (Kaiming) Initialization — for ReLU

**Formula:** `Var(W) = 2 / fan_in`

**Plain English:** ReLU kills exactly half of all activations (the negative ones). So the effective variance coming out of a ReLU layer is cut in half. He initialization compensates for this by doubling the target variance — using `2 / fan_in` instead of `2 / (fan_in + fan_out)`. This keeps signal strength stable through many ReLU layers.

In [ ]:
def get_first_layer_weights(init_fn_name):
    """Create a fresh MLP and return the first layer weights after applying init."""
    torch.manual_seed(0)
    net = MLP(input_dim=INPUT_DIM)
    if init_fn_name == 'default':
        pass  # PyTorch default (Kaiming uniform)
    elif init_fn_name == 'xavier_normal':
        for m in net.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    elif init_fn_name == 'he_normal':
        for m in net.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    elif init_fn_name == 'zeros':
        for m in net.modules():
            if isinstance(m, nn.Linear):
                nn.init.zeros_(m.weight)
    # Extract first layer weights
    w = net.layer1[0].weight.detach().numpy().flatten()
    return w

init_names   = ['zeros', 'default', 'xavier_normal', 'he_normal']
init_labels  = ['All Zeros (bad!)', 'PyTorch Default\n(Kaiming uniform)', 'Xavier Normal\n(for Sigmoid/Tanh)', 'He Normal\n(for ReLU)']
init_colors  = ['gray', 'steelblue', 'tomato', 'seagreen']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Weight Initialization: Distribution of First Layer Weights', fontsize=13, fontweight='bold')

for ax, name, label, color in zip(axes, init_names, init_labels, init_colors):
    weights = get_first_layer_weights(name)
    ax.hist(weights, bins=50, color=color, alpha=0.85, edgecolor='white')
    ax.set_title(label, fontweight='bold', fontsize=10)
    ax.set_xlabel('Weight value')
    ax.set_ylabel('Count')
    ax.axvline(0, color='black', linewidth=1, linestyle='--')
    std = weights.std()
    ax.text(0.05, 0.92, f'std={std:.4f}', transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### How to Read This Chart: Weight Initialization Distributions

Each panel shows a **histogram of the initial weight values** in the first linear layer of our MLP.

- **x-axis** = the weight value (how large or small each individual weight starts).
- **y-axis** = how many weights have that value.
- **The vertical dashed line at 0** shows the center.
- **std** = standard deviation of the weights — a measure of how spread out they are.

**What each initialization tells you:**
- **All Zeros** — the distribution is a single spike at 0. This is the bad case: symmetry is never broken, neurons never learn different things.
- **PyTorch Default (Kaiming uniform)** — weights are spread across a range, centered at zero. This is actually He initialization in uniform form. Good for ReLU networks.
- **Xavier Normal** — a normal distribution with variance `2 / (fan_in + fan_out)`. The spread is calibrated for sigmoid/tanh — slightly narrower than He, since those activations don't cut half the signal.
- **He Normal** — a normal distribution with variance `2 / fan_in`. Slightly wider than Xavier to compensate for ReLU's half-zeroing. This is the right choice for our ReLU network.

**Key insight:** Xavier keeps the **signal variance constant** assuming symmetric activations. He accounts for the fact that **ReLU cuts exactly half the activations** to zero, so variance needs to start 2× higher to come out the same on the other side.

---

## Section 7 — Training Loop

### The Full Training Loop

Training a PyTorch model involves repeating this cycle for every **epoch** (one full pass through the training data):

1. **Load a batch** — DataLoader feeds `batch_size` examples at a time.
2. **Forward pass** — call `model(x)` to get predictions.
3. **Compute loss** — compare predictions to true labels using the loss function.
4. **Backward pass** — call `loss.backward()` to compute gradients.
5. **Update weights** — call `optimizer.step()` to adjust all parameters using their gradients.
6. **Zero gradients** — call `optimizer.zero_grad()` to clear gradients before the next batch (otherwise they accumulate).

### Why Adam?

**Adam** (Adaptive Moment Estimation) is the most popular optimizer for deep learning. It adapts the learning rate **per parameter** based on the history of gradients. Parameters that receive large, consistent gradients get a smaller effective learning rate. Parameters that receive small or noisy gradients get a larger effective learning rate. This makes Adam very robust across different architectures and learning rate choices.

### Why CrossEntropyLoss?

For classification, **CrossEntropyLoss** is the standard choice. It measures how far the model's predicted class probabilities are from the true labels. It combines a softmax (to convert raw scores to probabilities) and negative log-likelihood into one step.

### Early Stopping

**Early stopping** monitors validation loss. If it doesn't improve for `patience` epochs in a row, training stops and the best model weights are restored. This prevents overfitting: the model doesn't keep training once it starts memorizing the training data.

In [ ]:
# ── Prepare DataLoaders ──
def make_loaders(X_train, y_train, X_val, y_val, batch_size=64):
    X_tr = torch.FloatTensor(X_train)
    y_tr = torch.LongTensor(y_train)
    X_vl = torch.FloatTensor(X_val)
    y_vl = torch.LongTensor(y_val)
    train_ds = TensorDataset(X_tr, y_tr)
    val_ds   = TensorDataset(X_vl, y_vl)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=512,        shuffle=False)
    return train_loader, val_loader

train_loader, val_loader = make_loaders(X_train_sc, y_train_np, X_test_sc, y_test_np)
print(f"Train batches : {len(train_loader)}  (batch size 64)")
print(f"Val batches   : {len(val_loader)}")

In [ ]:
def train_model(model, train_loader, val_loader, epochs=30, lr=1e-3, patience=5, verbose=True):
    """
    Full training loop with:
    - Adam optimizer
    - CrossEntropyLoss
    - Early stopping (patience)
    Returns: history dict with train/val loss and accuracy per epoch
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    best_val_loss  = float('inf')
    patience_count = 0
    best_state     = None

    for epoch in range(1, epochs + 1):
        # ── Training ──
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * X_batch.size(0)
            preds  = logits.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total   += X_batch.size(0)
        train_loss = running_loss / total
        train_acc  = correct / total

        # ── Validation ──
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                logits = model(X_batch)
                loss   = criterion(logits, y_batch)
                val_loss    += loss.item() * X_batch.size(0)
                preds  = logits.argmax(dim=1)
                val_correct += (preds == y_batch).sum().item()
                val_total   += X_batch.size(0)
        val_loss /= val_total
        val_acc   = val_correct / val_total

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        if verbose:
            print(f"Epoch {epoch:>3}/{epochs}  "
                  f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
                  f"train_acc={train_acc:.4f}  val_acc={val_acc:.4f}")

        # ── Early stopping ──
        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            import copy
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"  → Early stopping at epoch {epoch} (no improvement for {patience} epochs)")
                break

    # Restore best weights
    if best_state is not None:
        model.load_state_dict(best_state)
    return history


# Train the main MLP
torch.manual_seed(42)
main_model = MLP(input_dim=INPUT_DIM).to(device)
history = train_model(main_model, train_loader, val_loader, epochs=30, lr=1e-3, patience=5)

In [ ]:
# ── Plot learning curves ──
epochs_run = len(history['train_loss'])
ep_range   = range(1, epochs_run + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(ep_range, history['train_loss'], label='Train Loss',      color='steelblue',  linewidth=2)
ax1.plot(ep_range, history['val_loss'],   label='Validation Loss', color='tomato',     linewidth=2, linestyle='--')
ax1.set_title('Learning Curves — Loss', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.legend()
ax1.grid(True, alpha=0.4)

# Accuracy
ax2.plot(ep_range, history['train_acc'], label='Train Accuracy',      color='steelblue',  linewidth=2)
ax2.plot(ep_range, history['val_acc'],   label='Validation Accuracy', color='tomato',     linewidth=2, linestyle='--')
ax2.set_title('Learning Curves — Accuracy', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_ylim(0.5, 1.0)
ax2.legend()
ax2.grid(True, alpha=0.4)

plt.suptitle('MLP on Adult Income — Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

final_val_acc = history['val_acc'][-1]
print(f"Final validation accuracy: {final_val_acc:.4f}")

### How to Read This Chart: Learning Curves (Loss)

This chart shows the **cross-entropy loss** on training and validation data across all epochs.

- **x-axis** = epoch number (one epoch = one full pass through the training data).
- **y-axis** = loss value. **Lower is better.**
- **Blue line** = training loss — how well the model fits the data it was trained on.
- **Red dashed line** = validation loss — how well the model generalizes to data it has never seen.

**Patterns to look for:**
- **Both lines decrease together** → Good. The model is learning and generalizing well.
- **Train loss decreases but val loss increases** → Overfitting. The model memorizes training data but fails on new data.
- **Both lines stay high** → Underfitting. The model is too simple to capture the patterns.
- **Val loss stops improving for several epochs** → That's when early stopping triggers. The model saved at the best validation loss point.

---

### How to Read This Chart: Learning Curves (Accuracy)

Same structure as the loss chart, but showing **accuracy** (fraction of correct predictions).

- **y-axis** = accuracy from 0 to 1. **Higher is better.**
- **Blue line** = training accuracy.
- **Red dashed line** = validation accuracy.

**Patterns to look for:**
- **Large gap between blue and red** → The model is overfitting: it's much better on training data than on validation data.
- **Both converging to a high value** → Good fit.
- **Both converging to a low value** → Underfitting. Consider a larger model, more epochs, or better features.
- The validation accuracy is the number that matters for real-world performance.

---

## Section 8 — Diagnosing Underfitting, Overfitting, Good Fit

### What Are These Three Cases?

- **Underfitting:** The model is too simple to learn the patterns in the data. Both training and validation errors are high. Adding more capacity (more layers, more neurons) or training longer typically helps.

- **Overfitting:** The model is too complex for the amount of data. Training error is low but validation error is high and diverging. The model has essentially memorized the training set. Dropout, regularization, or more data typically helps.

- **Good fit (just right):** Both training and validation errors are low and converging. The model has learned the underlying pattern without memorizing noise.

We will train three models to demonstrate all three cases.

In [ ]:
# ── Tiny model (underfitting) ──
class TinyMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 4),
            nn.ReLU(),
            nn.Linear(4, 2)
        )
    def forward(self, x):
        return self.net(x)

# ── Huge model with NO dropout (overfitting) ──
class HugeMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 2048), nn.ReLU(),
            nn.Linear(2048, 2048),      nn.ReLU(),
            nn.Linear(2048, 1024),      nn.ReLU(),
            nn.Linear(1024, 1024),      nn.ReLU(),
            nn.Linear(1024, 512),       nn.ReLU(),
            nn.Linear(512, 2)
        )
    def forward(self, x):
        return self.net(x)

# ── Medium model with Dropout (good fit) — same as our main MLP ──
torch.manual_seed(42)
tiny_model = TinyMLP(INPUT_DIM).to(device)
huge_model = HugeMLP(INPUT_DIM).to(device)
good_model = MLP(INPUT_DIM).to(device)

print("Training tiny model (should underfit)...")
hist_tiny = train_model(tiny_model, train_loader, val_loader, epochs=30, patience=30, verbose=False)

print("Training huge model (should overfit)...")
hist_huge = train_model(huge_model, train_loader, val_loader, epochs=30, patience=30, verbose=False)

print("Training good model (Dropout + BatchNorm)...")
hist_good = train_model(good_model, train_loader, val_loader, epochs=30, patience=30, verbose=False)

print("Done.")

In [ ]:
# ── Plot all three side by side ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

titles  = ['Underfitting\n(Tiny MLP: 4 neurons)', 'Overfitting\n(Huge MLP: no Dropout)', 'Good Fit\n(Medium MLP + Dropout)']
hists   = [hist_tiny, hist_huge, hist_good]

for ax, title, hist in zip(axes, titles, hists):
    ep = range(1, len(hist['train_loss']) + 1)
    ax.plot(ep, hist['train_loss'], label='Train Loss', color='steelblue', linewidth=2)
    ax.plot(ep, hist['val_loss'],   label='Val Loss',   color='tomato',    linewidth=2, linestyle='--')
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.4)
    final_tr = hist['train_loss'][-1]
    final_vl = hist['val_loss'][-1]
    ax.text(0.05, 0.92, f"Final train: {final_tr:.3f}\nFinal val:   {final_vl:.3f}",
            transform=ax.transAxes, fontsize=8,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Diagnosing Fit: Underfitting vs Overfitting vs Good Fit', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### How to Read This Chart: Diagnosing Fit

Three side-by-side panels, each showing the same type of learning curve plot (train loss vs val loss). The panels differ only in which model was trained.

**Left panel — Underfitting (Tiny MLP: only 4 neurons):**
- Both train loss and val loss are **high** and barely decrease.
- The lines converge, but to a high loss value — neither line is good.
- Interpretation: the model is **too simple** to capture the patterns. It doesn't even fit the training data well.
- Fix: use a bigger model (more layers and neurons).

**Middle panel — Overfitting (Huge MLP with no Dropout):**
- Train loss drops very fast to a low value.
- Val loss may decrease initially, then **flattens or rises** while train loss keeps falling.
- The gap between the two lines grows over time.
- Interpretation: the model is memorizing training examples rather than learning general patterns.
- Fix: add Dropout, reduce model size, use more data, or add early stopping.

**Right panel — Good Fit (Medium MLP + Dropout + BatchNorm):**
- Both lines decrease steadily.
- The **gap between train and val loss is small**.
- Both converge to a reasonably low loss.
- Interpretation: the model has learned the general patterns without memorizing noise.
- This is the ideal outcome.

---

## Section 9 — BatchNorm and Dropout Effects

### What Do These Techniques Do?

**Batch Normalization (BatchNorm):**
After each linear layer, normalizes the activations to have mean ≈ 0 and standard deviation ≈ 1 across the current batch. Then applies learned scale (γ) and shift (β) parameters.

**Why it helps:**
- Reduces **internal covariate shift** — the tendency of each layer's input distribution to change as earlier layers update, which makes training unstable.
- Acts as a mild regularizer (slight noise from batch statistics).
- Allows higher learning rates.

**Dropout:**
During training, randomly sets a fraction `p` of neuron outputs to zero on every forward pass. The dropped neurons are different each batch.

**Why it helps:**
- Prevents neurons from **co-adapting** — no neuron can rely on a specific other neuron always being present.
- Forces the network to learn redundant representations.
- During inference, Dropout is disabled and all neurons contribute.

We now compare four variants to see the effect of each technique.

In [ ]:
class MLPVariant(nn.Module):
    """Configurable MLP: enable/disable BatchNorm and Dropout independently."""
    def __init__(self, input_dim, use_bn=True, use_dropout=True, dropout_rate=0.3):
        super().__init__()
        def block(in_f, out_f):
            layers = [nn.Linear(in_f, out_f)]
            if use_bn:
                layers.append(nn.BatchNorm1d(out_f))
            layers.append(nn.ReLU())
            if use_dropout:
                layers.append(nn.Dropout(dropout_rate))
            return nn.Sequential(*layers)
        self.net = nn.Sequential(
            block(input_dim, 256),
            block(256, 128),
            block(128, 64),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        return self.net(x)


configs = [
    ('No regularization',   dict(use_bn=False, use_dropout=False)),
    ('Dropout only',        dict(use_bn=False, use_dropout=True)),
    ('BatchNorm only',      dict(use_bn=True,  use_dropout=False)),
    ('Dropout + BatchNorm', dict(use_bn=True,  use_dropout=True)),
]

reg_results = []
for label, cfg in configs:
    torch.manual_seed(42)
    m = MLPVariant(INPUT_DIM, **cfg).to(device)
    h = train_model(m, train_loader, val_loader, epochs=20, patience=20, verbose=False)
    final_tr_acc = h['train_acc'][-1]
    final_vl_acc = h['val_acc'][-1]
    gap = final_tr_acc - final_vl_acc
    reg_results.append({'Config': label,
                        'Train Acc': final_tr_acc,
                        'Val Acc':   final_vl_acc,
                        'Gap (overfit)': gap})
    print(f"{label:<30}: train={final_tr_acc:.4f}  val={final_vl_acc:.4f}  gap={gap:.4f}")

reg_df = pd.DataFrame(reg_results)
print()
print(reg_df.to_string(index=False))

### How to Read This Table: Regularization Comparison

Each row is a different configuration of our MLP, trained for the same number of epochs on the same data.

| Column | What it means |
|--------|---------------|
| **Config** | Which regularization techniques were active |
| **Train Acc** | Accuracy on the training set — measures how well the model fits training data |
| **Val Acc** | Accuracy on the held-out test set — measures generalization |
| **Gap (overfit)** | Train Acc − Val Acc. A large gap = overfitting |

**What to look for:**

- **No regularization** — typically shows the highest Train Acc but the largest Gap. The model memorizes training data.
- **Dropout only** — reduces the gap because neurons can't co-adapt, but without BatchNorm, training may be less stable.
- **BatchNorm only** — stabilizes training and can slightly reduce overfitting through the noise in batch statistics, but less explicit regularization.
- **Dropout + BatchNorm** — usually the best combination: stable training, lowest gap, good generalization.

**Why this matters:**
- The goal is a **high Val Acc with a small Gap**.
- High Train Acc alone is not useful — a model that gets 99% on training data but 70% on new data is unreliable.

---

## Section 10 — MLP vs XGBoost on Tabular Data

### Why Compare These Two?

Neural networks dominate on **unstructured data** (images, text, audio) where the raw input has spatial or sequential structure that MLPs and specialized architectures can exploit. But on **tabular data** — the kind you find in spreadsheets and databases — the picture is different.

**The key finding from research:**  
On tabular data, tree-based models (XGBoost, LightGBM, CatBoost) **almost always outperform MLP** without careful tuning. This is a well-documented empirical finding.

**Why?**
- Trees handle **mixed data types** (numerical + categorical) naturally.
- Trees are **invariant to feature scaling** — no need for StandardScaler.
- Trees handle **feature interactions** automatically through splits.
- MLPs on tabular data require significant tuning: learning rate, architecture, regularization, initialization.

We will now benchmark both on the Adult Income dataset with the same preprocessing and compare accuracy, F1, and speed.

In [ ]:
# ── Train XGBoost (or GradientBoosting if XGBoost not installed) ──
print("Training tree-based model...")
t0 = time.time()
if XGBOOST_AVAILABLE:
    tree_model = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        use_label_encoder=False, eval_metric='logloss',
        random_state=42, verbosity=0
    )
    tree_label = 'XGBoost'
else:
    tree_model = GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42
    )
    tree_label = 'GradientBoosting (sklearn)'

tree_model.fit(X_train_np, y_train_np)  # Note: trees don't need scaling
tree_train_time = time.time() - t0

t0 = time.time()
tree_preds = tree_model.predict(X_test_np)
tree_inf_time = time.time() - t0

tree_acc = accuracy_score(y_test_np, tree_preds)
tree_f1  = f1_score(y_test_np, tree_preds)
print(f"{tree_label} — Acc: {tree_acc:.4f}, F1: {tree_f1:.4f}, "
      f"Train time: {tree_train_time:.2f}s, Inf time: {tree_inf_time:.4f}s")

In [ ]:
# ── Evaluate our trained MLP ──
main_model.eval()
X_test_t = torch.FloatTensor(X_test_sc).to(device)

t0 = time.time()
with torch.no_grad():
    mlp_logits = main_model(X_test_t)
    mlp_preds  = mlp_logits.argmax(dim=1).cpu().numpy()
mlp_inf_time = time.time() - t0

mlp_acc = accuracy_score(y_test_np, mlp_preds)
mlp_f1  = f1_score(y_test_np, mlp_preds)

# ── Comparison table ──
comp_df = pd.DataFrame([
    {'Model': 'MLP (PyTorch)',  'Accuracy': mlp_acc,  'F1 Score': mlp_f1,  'Inference Time (s)': mlp_inf_time},
    {'Model': tree_label,       'Accuracy': tree_acc, 'F1 Score': tree_f1, 'Inference Time (s)': tree_inf_time},
])
print(comp_df.to_string(index=False))

### How to Read This Table: MLP vs XGBoost Comparison

| Column | What it measures |
|--------|------------------|
| **Accuracy** | Fraction of all predictions that are correct |
| **F1 Score** | Harmonic mean of precision and recall. Better metric for imbalanced datasets — it penalizes models that only predict one class |
| **Inference Time** | How long it takes to predict on the full test set |

**Expected outcome:**

XGBoost will likely match or slightly outperform the MLP on this dataset, despite far simpler code and no need for:
- Feature scaling
- Careful architecture design
- Hyperparameter tuning of dropout / learning rate
- Early stopping

**This is not a criticism of neural networks.** MLPs become dominant when:
- The data is **images, text, or audio** — where spatial/sequential structure matters.
- You have **millions of training examples** — trees stop improving past a certain size, networks keep improving.
- You need **end-to-end learning** — feeding raw pixels directly rather than hand-crafted features.

**On tabular data:** Always try XGBoost or LightGBM first. If you want MLP to compete, invest in tuning or try newer architectures like TabNet or FT-Transformer.

---

## Section 11 — Autoencoder on MNIST

### What Is an Autoencoder?

**An autoencoder learns to compress data into a bottleneck representation and then reconstruct it. The compression forces the network to learn the most important features.**

An autoencoder is a neural network with two connected parts:

**Encoder:** Takes the input and progressively compresses it into a small vector called the **latent space** or **bottleneck**.

**Decoder:** Takes the bottleneck vector and progressively reconstructs the original input.

The model is trained to **minimize reconstruction error** — to output something as close to the original input as possible.

**Why is this useful?**
- The bottleneck representation captures the **most essential structure** in the data.
- It can be used for **dimensionality reduction**, **anomaly detection**, **data compression**, and **generative models**.
- Crucially, the model learns **without labels** — it is unsupervised.

### Architecture

```
Input (784)  ──► Encoder: 256 → 128 → 64 → 32 → [Bottleneck: 8]
                 Decoder: 8 → 32 → 64 → 128 → 256 → Output (784)
```

MNIST images are 28×28 = 784 pixels. We compress them to just **8 numbers** — a 98% compression rate — and then reconstruct the original image. The 8 numbers must capture the essence of each digit.

In [ ]:
# ── Load MNIST ──
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))  # Flatten 28x28 to 784
])

mnist_train = torchvision.datasets.MNIST(
    root='./data', train=True,  download=True, transform=mnist_transform
)
mnist_test  = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=mnist_transform
)

mnist_train_loader = DataLoader(mnist_train, batch_size=256, shuffle=True)
mnist_test_loader  = DataLoader(mnist_test,  batch_size=512, shuffle=False)

print(f"MNIST train samples: {len(mnist_train)}")
print(f"MNIST test  samples: {len(mnist_test)}")
print(f"Input dimension: 784 (28×28 pixels, flattened)")

In [ ]:
class Autoencoder(nn.Module):
    """
    Feedforward Autoencoder for MNIST.
    Encoder: 784 → 256 → 128 → 64 → 32 → 8  (bottleneck)
    Decoder: 8 → 32 → 64 → 128 → 256 → 784
    """
    def __init__(self, bottleneck_dim=8):
        super(Autoencoder, self).__init__()

        self.encoder = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64),  nn.ReLU(),
            nn.Linear(64, 32),   nn.ReLU(),
            nn.Linear(32, bottleneck_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 32),  nn.ReLU(),
            nn.Linear(32, 64),              nn.ReLU(),
            nn.Linear(64, 128),             nn.ReLU(),
            nn.Linear(128, 256),            nn.ReLU(),
            nn.Linear(256, 784),            nn.Sigmoid()  # Output in [0,1] to match pixel values
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon

    def encode(self, x):
        return self.encoder(x)


torch.manual_seed(42)
ae_model = Autoencoder(bottleneck_dim=8).to(device)
print(ae_model)

# Parameter count
total = sum(p.numel() for p in ae_model.parameters())
print(f"\nTotal parameters: {total:,}")

In [ ]:
# ── Train the Autoencoder ──
ae_criterion = nn.MSELoss()
ae_optimizer = optim.Adam(ae_model.parameters(), lr=1e-3)

ae_history = {'train_loss': []}

EPOCHS_AE = 20
print(f"Training Autoencoder for {EPOCHS_AE} epochs...\n")

for epoch in range(1, EPOCHS_AE + 1):
    ae_model.train()
    running_loss, total_samples = 0.0, 0
    for X_batch, _ in mnist_train_loader:          # Labels not used!
        X_batch = X_batch.to(device)
        ae_optimizer.zero_grad()
        X_recon = ae_model(X_batch)
        loss    = ae_criterion(X_recon, X_batch)   # Reconstruct = original
        loss.backward()
        ae_optimizer.step()
        running_loss   += loss.item() * X_batch.size(0)
        total_samples  += X_batch.size(0)
    epoch_loss = running_loss / total_samples
    ae_history['train_loss'].append(epoch_loss)
    print(f"Epoch {epoch:>3}/{EPOCHS_AE}  Reconstruction Loss = {epoch_loss:.6f}")

print("\nAutoencoder training complete.")

In [ ]:
# ── Plot reconstruction loss ──
plt.figure(figsize=(9, 4))
plt.plot(range(1, EPOCHS_AE + 1), ae_history['train_loss'], color='steelblue', linewidth=2.5, marker='o', markersize=4)
plt.title('Autoencoder Reconstruction Loss (MSE) Over Training', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

### How to Read This Chart: Autoencoder Reconstruction Loss

This shows the **Mean Squared Error (MSE)** between the original MNIST images and the reconstructed images, averaged over all training examples, per epoch.

- **x-axis** = training epoch.
- **y-axis** = MSE reconstruction loss. **Lower is better.**
- MSE measures the average squared pixel difference between original and reconstruction. A value of 0 = perfect reconstruction.

**What to look for:**
- The loss should decrease smoothly and eventually plateau — this is the typical learning curve for autoencoders.
- An early sharp drop means the model quickly learns coarse structure (rough digit shapes).
- Slower improvement in later epochs corresponds to learning finer details.
- If the loss plateaus very high, the bottleneck dimension (8) may be too small to encode the necessary information.

In [ ]:
# ── Visualize original vs reconstructed ──
ae_model.eval()

# Get a batch from test set
test_batch_x, test_batch_y = next(iter(mnist_test_loader))
test_batch_x = test_batch_x.to(device)

with torch.no_grad():
    recons = ae_model(test_batch_x).cpu().numpy()

originals = test_batch_x.cpu().numpy()

n_show = 8
fig, axes = plt.subplots(2, n_show, figsize=(16, 4))
fig.suptitle('Original (top) vs Reconstructed (bottom) MNIST digits', fontsize=13, fontweight='bold')

for i in range(n_show):
    # Original
    axes[0, i].imshow(originals[i].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    axes[0, i].axis('off')
    axes[0, i].set_title(f'Label: {test_batch_y[i].item()}', fontsize=8)

    # Reconstruction
    axes[1, i].imshow(recons[i].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original',       rotation=90, labelpad=20, fontsize=10)
axes[1, 0].set_ylabel('Reconstructed',  rotation=90, labelpad=20, fontsize=10)
plt.tight_layout()
plt.show()

### How to Read This Chart: Original vs Reconstructed

Two rows of 8 grayscale images:
- **Top row** = original MNIST digit images from the test set.
- **Bottom row** = the autoencoder's reconstruction of those same images.
- **Labels** = the actual digit class (0–9), for reference only — the autoencoder was not trained with these labels.

**What to look for:**
- If reconstructions look similar to originals (correct digit shape, clear strokes), the autoencoder has learned a useful compressed representation.
- **Slight blurriness** is normal and expected. The bottleneck of only 8 dimensions cannot perfectly capture every pixel — it must throw away some detail. The model keeps the globally important structure and loses fine details.
- **Very blurry** = bottleneck too small. The model cannot encode enough information.
- **Nearly identical to original** = bottleneck too large. The model might be memorizing rather than truly compressing.
- Wrong digit shapes in reconstructions would indicate the model has not learned to represent digits properly.

In [ ]:
# ── Extract bottleneck representations and visualize with PCA ──

# Collect all test set latent vectors
all_z, all_labels = [], []
ae_model.eval()
with torch.no_grad():
    for X_batch, y_batch in mnist_test_loader:
        z = ae_model.encode(X_batch.to(device))
        all_z.append(z.cpu().numpy())
        all_labels.append(y_batch.numpy())

all_z      = np.vstack(all_z)
all_labels = np.concatenate(all_labels)

# Reduce to 2D with PCA
pca = PCA(n_components=2, random_state=42)
z_2d = pca.fit_transform(all_z)

print(f"Latent space shape: {all_z.shape}  →  PCA 2D shape: {z_2d.shape}")
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# Scatter plot
plt.figure(figsize=(10, 8))
scatter = plt.scatter(z_2d[:, 0], z_2d[:, 1],
                      c=all_labels, cmap='tab10',
                      s=4, alpha=0.6)
cbar = plt.colorbar(scatter, ticks=range(10))
cbar.set_label('Digit Class', fontsize=11)
cbar.set_ticklabels([str(i) for i in range(10)])
plt.title('Autoencoder Latent Space — 8D → PCA 2D\nColored by Digit Class (no labels used during training)',
          fontweight='bold')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### How to Read This Chart: Autoencoder Latent Space

This is a **2D scatter plot** of the autoencoder's 8-dimensional bottleneck representations, reduced to 2D using PCA.

- **Each dot** represents one MNIST test image — its position in 2D is determined by its 8-dim latent code, projected down.
- **Colors** correspond to the true digit class (0–9). The autoencoder was **never given these labels** — they are applied here just for visualization.
- **Proximity** = similarity in latent space. Dots close together = images the model encodes similarly.

**What to look for:**
- **Distinct color clusters** → the autoencoder has learned to separate different digit classes in its internal representation, even though no label supervision was provided. This is **representation learning** — structure emerging from reconstruction alone.
- **Some clusters overlap** → visually similar digits (e.g., 4 and 9, or 3 and 8) may share similar latent codes, because they look alike.
- **Tight, well-separated clusters** → the bottleneck has found a compact, organized space. A linear classifier trained on these 8D latent vectors would perform well.
- **One big blob with no structure** → the bottleneck isn't forming meaningful representations — possibly the bottleneck is too small or training is insufficient.

**Why this matters:**
This demonstrates that neural networks can discover structure in data **without any human-provided labels** — purely through the pressure of learning to reconstruct inputs with limited capacity.

---

## Section 12 — Autoencoder for Anomaly Detection (Preview)

### The Key Idea

An autoencoder trained on **only one class** will learn to reconstruct that class well. When shown examples from **a different class**, it will reconstruct them poorly — resulting in a high reconstruction error.

**This reconstruction error becomes a detector for anomalies** (outliers, unusual examples, or data from different distributions).

This approach is widely used in:
- **Fraud detection** — train on normal transactions, flag high-error transactions as anomalies.
- **Manufacturing defect detection** — train on good parts, detect bad ones.
- **Intrusion detection** — train on normal network traffic, detect attacks.
- **Medical imaging** — train on healthy scans, detect unusual findings.

Here, we will train on digit '0' only, then compute reconstruction error for all 10 digits.

In [ ]:
# ── Prepare one-class dataset (digit '0' only) ──
# Extract all digit-0 examples from training set
all_train_x = mnist_train.data.float() / 255.0  # Normalize to [0,1]
all_train_y = mnist_train.targets

zero_mask  = (all_train_y == 0)
zero_x     = all_train_x[zero_mask].view(-1, 784)  # Flatten
print(f"Training examples for digit 0: {zero_x.shape[0]}")

# Create DataLoader
zero_ds     = TensorDataset(zero_x)
zero_loader = DataLoader(zero_ds, batch_size=128, shuffle=True)

# ── Train one-class autoencoder ──
torch.manual_seed(42)
ae_oneclass = Autoencoder(bottleneck_dim=8).to(device)
ae_opt_oc   = optim.Adam(ae_oneclass.parameters(), lr=1e-3)

EPOCHS_OC = 20
print(f"\nTraining one-class autoencoder on digit '0' for {EPOCHS_OC} epochs...")
for epoch in range(1, EPOCHS_OC + 1):
    ae_oneclass.train()
    total_loss, total_n = 0.0, 0
    for (X_batch,) in zero_loader:
        X_batch = X_batch.to(device)
        ae_opt_oc.zero_grad()
        loss = nn.MSELoss()(ae_oneclass(X_batch), X_batch)
        loss.backward()
        ae_opt_oc.step()
        total_loss += loss.item() * X_batch.size(0)
        total_n    += X_batch.size(0)
    if epoch % 5 == 0 or epoch == 1:
        print(f"  Epoch {epoch:>3}/{EPOCHS_OC}  Loss = {total_loss/total_n:.6f}")
print("Done.")

In [ ]:
# ── Compute per-digit reconstruction errors on test set ──
all_test_x = mnist_test.data.float() / 255.0
all_test_y = mnist_test.targets.numpy()
all_test_x_flat = all_test_x.view(-1, 784)

ae_oneclass.eval()
errors_by_digit = {d: [] for d in range(10)}

BATCH = 256
with torch.no_grad():
    for start in range(0, len(all_test_x_flat), BATCH):
        X_b = all_test_x_flat[start:start+BATCH].to(device)
        y_b = all_test_y[start:start+BATCH]
        X_r = ae_oneclass(X_b)
        # Per-sample MSE
        per_sample_err = ((X_r - X_b) ** 2).mean(dim=1).cpu().numpy()
        for err, label in zip(per_sample_err, y_b):
            errors_by_digit[int(label)].append(float(err))

# Print mean error per digit
print("Mean reconstruction error per digit class:")
print(f"{'Digit':<8} {'Mean MSE':>12} {'Note'}")
print("-" * 40)
for d in range(10):
    mean_err = np.mean(errors_by_digit[d])
    note = "<-- trained on this" if d == 0 else ""
    print(f"{d:<8} {mean_err:>12.6f}  {note}")

In [ ]:
# ── Violin / Box plot of errors per digit ──
error_records = []
for d in range(10):
    for e in errors_by_digit[d]:
        error_records.append({'Digit': str(d), 'MSE': e})
error_df = pd.DataFrame(error_records)

plt.figure(figsize=(14, 6))
palette = ['tomato' if d == '0' else 'steelblue' for d in sorted(error_df['Digit'].unique(), key=int)]
sns.boxplot(data=error_df, x='Digit', y='MSE', order=[str(i) for i in range(10)],
            palette=palette, showfliers=False)
plt.title('Reconstruction Error per Digit Class\n(Autoencoder trained on digit "0" only)',
          fontweight='bold', fontsize=13)
plt.xlabel('Digit Class')
plt.ylabel('MSE (reconstruction error)')
plt.text(0, error_df[error_df['Digit']=='0']['MSE'].quantile(0.75) * 1.1,
         'Trained class\n(low error)', ha='center', color='tomato', fontsize=9, fontweight='bold')
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

### How to Read This Chart: Anomaly Detection via Reconstruction Error

This is a **box plot** showing the distribution of MSE reconstruction errors for each digit class (0–9).

- **x-axis** = digit class (0 through 9).
- **y-axis** = MSE reconstruction error per image. **Higher error = worse reconstruction.**
- **Each box** shows the interquartile range (middle 50% of error values) for that digit class. The horizontal line inside the box is the median.
- **Red (digit 0)** = the class the autoencoder was trained on.
- **Blue (digits 1–9)** = classes the autoencoder has never seen.

**What to look for:**
- Digit **0 should have clearly lower error** than other digits — the model was trained to reconstruct zeros, so it does this well.
- All other digits should have **higher error** — the model was never trained on them, so its encoder compresses them through a representation optimized for zeros, and the decoder reconstructs something that looks vaguely like a zero.
- Some digits (like **6** or **8**) may have lower error than others if they visually resemble a zero (circular shape). This is a feature, not a bug — it shows the model is using visual similarity.

**In a real anomaly detection system:**
- You would choose a **threshold** on the reconstruction error.
- Examples with error **above the threshold** are flagged as anomalies.
- The ROC curve (not shown here) would help you choose the threshold that best balances false positives and false negatives.

---

## Section 13 — Summary and Key Takeaways

### What We Covered

This notebook walked through the complete landscape of feedforward neural networks — from first principles to practical applications.

---

### When to Use MLP vs Tree Models

| Situation | Best Choice | Why |
|-----------|-------------|-----|
| Tabular data (spreadsheets, databases) | XGBoost / LightGBM | Fewer assumptions, no scaling needed, handles mixed types, robust out of the box |
| Images | CNN (not plain MLP) | Spatial structure — convolutions exploit pixel locality |
| Text / Sequences | Transformer / RNN | Sequential structure — attention or recurrence exploits order |
| Audio | CNN / RNN on spectrograms | Frequency and time structure |
| Millions of examples | Deep MLP / specialized arch | Networks scale better than trees beyond a certain data size |
| Unsupervised representation learning | Autoencoder / VAE | Self-supervised compression and reconstruction |

**Rule of thumb:** If your data lives in a spreadsheet and you have fewer than ~1 million rows, start with XGBoost. It will probably win with less effort. If you care about representation learning, need end-to-end training, or are working with images/text/audio — neural networks are the right tool.

---

### The MLP Training Checklist

Before training any MLP, make sure you have addressed:

| Item | Recommended Setting | Why |
|------|---------------------|-----|
| **Activation function** | ReLU (or Leaky ReLU) for hidden layers | Avoids vanishing gradients |
| **Weight initialization** | He (Kaiming) for ReLU, Xavier for Sigmoid/Tanh | Keeps signal stable at start |
| **Batch Normalization** | After each Linear layer, before activation | Stabilizes training, allows higher LR |
| **Dropout** | 0.2–0.5 for hidden layers | Prevents co-adaptation and overfitting |
| **Optimizer** | Adam (lr ≈ 1e-3 to start) | Adaptive, robust across problems |
| **Loss function** | CrossEntropyLoss (classification), MSELoss (regression) | Mathematically appropriate per task |
| **Learning rate tuning** | Try 1e-2, 1e-3, 1e-4 | Single most impactful hyperparameter |
| **Early stopping** | patience = 5–10 epochs | Prevents overfitting, saves time |
| **Feature scaling** | StandardScaler | MLPs are sensitive to input scale |

---

### Common Mistakes to Avoid

1. **Using Sigmoid activations in hidden layers of deep networks.** Gradient vanishes exponentially with depth. Use ReLU.

2. **No weight initialization strategy.** PyTorch defaults to Kaiming uniform (He), which is correct for ReLU — don't override it without a reason.

3. **No early stopping.** Running training to a fixed number of epochs without monitoring validation loss — you will overfit.

4. **Trusting training accuracy instead of validation accuracy.** A model that achieves 99% training accuracy but 70% validation accuracy is useless in production.

5. **Not scaling features.** Neural networks are highly sensitive to the scale of inputs. Always apply StandardScaler (or MinMaxScaler) before feeding data to an MLP.

6. **Forgetting `model.eval()` during inference.** Without this, Dropout is still active and BatchNorm uses batch statistics instead of running statistics — predictions will be inconsistent and wrong.

7. **Accumulating gradients.** Always call `optimizer.zero_grad()` before each backward pass. Otherwise, gradients from previous batches add up and corrupt the update.

---

### Architecture Design Guidelines

- **Start wide** — it is easier to add regularization to a wide network than to add capacity to a narrow one. A network that underfits is less useful as a starting point.

- **Add depth carefully** — going from 2 to 3 hidden layers is usually safe. Going from 3 to 10 requires careful initialization, BatchNorm, and appropriate activations. Deep is not always better.

- **Funnel shape** — progressively narrowing layers (512 → 256 → 128 → 64) is a common and effective pattern. Each layer compresses features into a more abstract representation.

- **More layers ≠ better.** The Universal Approximation Theorem says a single hidden layer with enough neurons can approximate any function. Depth helps with **efficiency** (fewer parameters for the same expressiveness) but is not a guarantee of better performance.

- **Match architecture to problem.** Tabular: 2–4 hidden layers. Image classification: CNN backbone. Text: Transformer. Sequence: LSTM or Transformer. Don't force a plain MLP onto everything.

---

### Autoencoder Key Points

- **Training objective = reconstruction** — no labels needed.
- **Bottleneck size controls the compression ratio.** Too small = blurry reconstructions. Too large = no compression happens.
- **Latent space organization emerges from structure in the data** — the model discovers clusters without supervision.
- **Reconstruction error = anomaly score** — train on one class, detect others by high error.
- **Next step from autoencoders:** Variational Autoencoders (VAEs) — add a probabilistic latent space for generation. These are covered in a separate notebook.

---

### What to Learn Next

| Topic | What it is |
|-------|------------|
| **Convolutional Neural Networks (CNNs)** | MLPs specialized for images — spatial weight sharing |
| **Recurrent Neural Networks (RNNs / LSTMs)** | MLPs for sequences — temporal memory |
| **Transformers** | Attention-based architecture powering modern NLP and vision |
| **Variational Autoencoders (VAEs)** | Generative autoencoders with probabilistic latent space |
| **Batch size effects** | How batch size interacts with BatchNorm and generalization |
| **Learning rate schedules** | CosineAnnealing, ReduceLROnPlateau, warmup |
| **Gradient clipping** | Preventing exploding gradients in very deep or recurrent networks |

---